In [79]:
import pandas as pd
import numpy as np

renewal_calls = pd.read_csv(
    "../../data/01_raw/raw_renewal_calls.csv",
    low_memory=False
)

renewal_calls.columns = (
    renewal_calls.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

renewal_calls = renewal_calls.drop_duplicates()

In [80]:
renewal_calls.shape

(157722, 41)

In [81]:
renewal_calls.info()
renewal_calls.head()

<class 'pandas.DataFrame'>
Index: 157722 entries, 0 to 186533
Data columns (total 41 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   call_id                                            157722 non-null  float64
 1   call_direction                                     157722 non-null  str    
 2   co_ref                                             153269 non-null  str    
 3   call_date                                          157722 non-null  str    
 4   churn_category                                     7901 non-null    str    
 5   complaint_category                                 18994 non-null   str    
 6   customer_reaction_category                         22986 non-null   str    
 7   agent_renewal_pitch_category                       53737 non-null   str    
 8   customer_renewal_response_category                 54311 non-null   str    
 9   agent_res

,call_id,call_direction,co_ref,call_date,churn_category,complaint_category,customer_reaction_category,agent_renewal_pitch_category,customer_renewal_response_category,agent_response_category,...,customer_response,desire_to_cancel,discount_offered,justification_category,reason_for_renewal_category,agent_response_to_cancel_category,argument_that_convinced_customer_to_stay_category,analysed_call,call_number,call_year
0,5.950000e+11,Outbound,UB0899,29-01-2025,NaN,NaN,Not Mentioned,Discussion / Introduction / Inquiry,Discount and Offer,Discount and Offer,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,3,2025
1,5.970000e+11,OUT_BOUND,HN5141,26-02-2025,NaN,NaN,NaN,Price and Cost,Agreement,Customer Communication,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,2,2025
2,5.950000e+11,Outbound,BP5009,24-01-2025,NaN,NaN,NaN,Expiration / Due,Agreement,Accreditation and Certification,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,1,2025
3,6.520000e+11,OUT_BOUND,XP8119,09-06-2025,NaN,NaN,NaN,Auto / Automatic,Agreement,Accreditation and Certification,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,1,2025
4,5.370000e+11,Outbound,ZL7978,20-08-2024,NaN,NaN,NaN,NaN,NaN,NaN,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,28,2024


In [82]:
renewal_calls = renewal_calls.drop_duplicates()
print("Shape after duplicate removal:", renewal_calls.shape)

Shape after duplicate removal: (157722, 41)


In [83]:
missing_df = (
    renewal_calls.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index()
)

missing_df.columns = ["column", "missing_percent"]
missing_df.head(50)

,column,missing_percent
0,unnamed:_20,100.000000
1,argument_that_convinced_customer_to_stay_category,98.847339
2,agent_response_to_cancel_category,96.876149
3,churn_category,94.990553
4,justification_category,91.706927
5,reason_for_renewal_category,89.769975
6,complaint_category,87.957292
7,customer_reaction_category,85.426256
8,agent_renewal_pitch_category,65.929293
9,agent_response_category,65.775859


In [84]:
missing_pct = renewal_calls.isnull().mean()

drop_cols = missing_pct[missing_pct > 0.80].index.tolist()

print("Columns to drop (>80% missing):")
print(drop_cols)

renewal_calls = renewal_calls.drop(columns=drop_cols)

print("Shape after dropping:", renewal_calls.shape)

Columns to drop (>80% missing):
['churn_category', 'complaint_category', 'customer_reaction_category', 'unnamed:_20', 'justification_category', 'reason_for_renewal_category', 'agent_response_to_cancel_category', 'argument_that_convinced_customer_to_stay_category']
Shape after dropping: (157722, 33)


In [85]:
cat_cols = renewal_calls.select_dtypes(include="object").columns.tolist()
num_cols = renewal_calls.select_dtypes(include=np.number).columns.tolist()

print("Categorical cols:", len(cat_cols))
print("Numeric cols:", len(num_cols))

Categorical cols: 29
Numeric cols: 4


C:\Users\NuluShreya\AppData\Local\Temp\ipykernel_43088\2597350013.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = renewal_calls.select_dtypes(include="object").columns.tolist()


In [86]:
# -------------------------------
# SMART NULL HANDLING FOR ALL COLUMNS
# -------------------------------

# categorical columns
for col in cat_cols:
    if renewal_calls[col].isnull().sum() > 0:
        renewal_calls[col] = renewal_calls[col].fillna("Unknown")

# numeric columns
for col in num_cols:
    if renewal_calls[col].isnull().sum() == 0:
        continue

    unique_vals = renewal_calls[col].dropna().nunique()

    # binary columns -> fill 0
    if unique_vals <= 2:
        renewal_calls[col] = renewal_calls[col].fillna(0)

    # count-like columns
    elif any(word in col.lower() for word in ["count", "num", "calls", "attempts"]):
        renewal_calls[col] = renewal_calls[col].fillna(0)

    # duration / score / continuous metrics
    else:
        renewal_calls[col] = renewal_calls[col].fillna(
            renewal_calls[col].median()
        )

In [87]:
null_summary = (
    renewal_calls.isnull()
    .sum()
    .sort_values(ascending=False)
)

print(null_summary.head(20))
print("Total remaining nulls:", renewal_calls.isnull().sum().sum())

call_id                                  0
call_direction                           0
co_ref                                   0
call_date                                0
agent_renewal_pitch_category             0
customer_renewal_response_category       0
agent_response_category                  0
membership_renewal_decision              0
serious_complaint                        0
other_complaint                          0
discussion_on_price_increase             0
renewal_impact_due_to_price_increase     0
discount_or_waiver_requested             0
call_reschedule_request                  0
agent_flagged_membership_status_alert    0
agent_renewal_initiation                 0
explicit_competitor_mention              0
explicit_switching_intent                0
mentioned_competitors                    0
price_switching_mentioned                0
dtype: int64
Total remaining nulls: 0


In [88]:
print("Final Shape:", renewal_calls.shape)
print("Remaining duplicates:", renewal_calls.duplicated().sum())

# quick datatype check
renewal_calls.info()

Final Shape: (157722, 33)
Remaining duplicates: 131
<class 'pandas.DataFrame'>
Index: 157722 entries, 0 to 186533
Data columns (total 33 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   call_id                                157722 non-null  float64
 1   call_direction                         157722 non-null  str    
 2   co_ref                                 157722 non-null  str    
 3   call_date                              157722 non-null  str    
 4   agent_renewal_pitch_category           157722 non-null  str    
 5   customer_renewal_response_category     157722 non-null  str    
 6   agent_response_category                157722 non-null  str    
 7   membership_renewal_decision            157722 non-null  str    
 8   serious_complaint                      157722 non-null  str    
 9   other_complaint                        157722 non-null  str    
 10  discussion_on_price_

In [89]:
num_check_cols = renewal_calls.select_dtypes(include=np.number).columns[:6]
renewal_calls[num_check_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
call_id,157722.0,5.989673e+11,7.506546e+10,1.930000e+11,5.130000e+11,5.950000e+11,6.800000e+11,6.950000e+11
analysed_call,157722.0,5.599029e-01,4.964003e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
call_number,157722.0,1.151508e+02,7.275789e+02,1.000000e+00,2.000000e+00,4.000000e+00,8.000000e+00,7.313000e+03
call_year,157722.0,2.024619e+03,5.971657e-01,2.024000e+03,2.024000e+03,2.025000e+03,2.025000e+03,2.026000e+03


In [92]:
renewal_calls.to_csv(
    "../../data/02_processed/processed_renewal_calls.csv",
    index=False
)

print(" processed_renewal_calls.csv saved successfully")

 processed_renewal_calls.csv saved successfully
